# Part b: Regression on the Runge function

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # add Code/ so `src` can be imported

In [ ]:
import numpy as np
import pandas as pd 
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler
from src.data import runge, make_runge_split, design_matrix, scale_data, center_data, make_batches 
from src.regression import *
from src.functions import *
from src.plotting import *
from src.network import NeuralNetwork 
from src.optimizers import *

In [ ]:
#Recreate dataset from project 1 
X_train, X_test, y_train, y_test = make_runge_split(
    n = 100,
    sigma = 0.1,
    test_size = 0.2,
    seed = 2026)

# Design matrix of degree 8 (optimal degree from project 1) 
Xt_train = design_matrix(X_train, degree=8, intercept=False)
Xt_test = design_matrix(X_test, degree=8, intercept=False)

# Center and scale the data 
Xt_train, Xt_test, X_mean, X_std = scale_data(Xt_train, Xt_test )
y_train_c, y_test_c, y_mean = center_data(y_train, y_test)

#Fitting the ols 
theta = ols_fit(Xt_train, y_train_c)

y_pred_ols = Xt_test @ theta + y_mean 
mse_ols = mse(y_pred_ols, y_test)

print(f"OLS test MSE: {mse_ols:.6f}")

# Generate x-values for a smooth curve
x_plot = np.linspace(-1, 1, 300).reshape(-1, 1)

# Use the same transformation as for the training data
Xt_plot = design_matrix(x_plot, degree=8, intercept=False)
Xt_plot = (Xt_plot - X_mean) / X_std

# Predict using the fitted OLS model
y_ols_plot = Xt_plot @ theta + y_mean

# True Runge function
y_true = runge(x_plot)

fig, ax = line_plot(
    x_plot,
    { "True Runge function": y_true,
        "OLS, degree 8": y_ols_plot,
    },
    xlabel="$x$",
    ylabel="$y$",
    data=(X_train, y_train),
    name="b_ols_fit")


Notes on the plot:
- noise is 0.1, variance of this is 0.01, all models would get mse with minimum 0.01. 
- oscillations on edges is runge phenomenon. 
- peak is not as sharp because of degree 8. 

## Using the network 

### Network with 50 nodes 

In [ ]:
# scale the original data 
X_train_s, X_test_s, x_mean, x_std = scale_data(X_train, X_test)

net50 = NeuralNetwork(
    network_input_size=1,
    layer_output_sizes=[50, 1],
    activation_funcs=[sigmoid, identity],
    activation_ders=[sigmoid_der, identity_der],
    cost_fun=mse,
    cost_der=mse_der,
    optimizer=Adam(gamma=0.01),
    seed=2026)

history50 = net50.train(
    X_train_s, 
    y_train, 
    epochs=500,
    batch_size=14,
    rng=np.random.default_rng(2026))

yn50_plot = net50.predict((x_plot - x_mean)/x_std) #scaling 

fig, ax = line_plot(
    x_plot,
    {
        "True Runge function": y_true,
        "OLS, degree 8": y_ols_plot,
        "NN, 1×50 sigmoid": yn50_plot,
    },
    xlabel="$x$",
    ylabel="$y$",
    data=(X_train, y_train),
    name="b50_nn_vs_ols")


Notes on the plot:
- nn removes runge phenomenon -> not polynomial basis anymore 
- shaper peak
- drops at right edge?? (fewer training points? )

In [ ]:
epochs = np.arange(1, len(history50) + 1)
fig, ax = line_plot(epochs, {"Adam": history50},
                    xlabel="Epoch", ylabel="Training MSE",
                    name="b50_learning_curve")

Notes on the plot:
- Loss drops fast -> only need around 100 epoch for future runs
- levels around 0.02, close to noise floor. 

In [ ]:
y_pred_n50 = net50.predict(X_test_s)
mse_n50 = mse(y_pred_n50, y_test)
print(f"NN50 test MSE: {mse_n50:.6f}")

Notes on mse:
- few test points -> use different seed for final comparison 
- difference to ols is more emaningful when taking into consideration the noise. 

### Network with 100 nodes 

In [ ]:
net100 = NeuralNetwork(
    network_input_size=1,
    layer_output_sizes=[100, 1],
    activation_funcs=[sigmoid, identity],
    activation_ders=[sigmoid_der, identity_der],
    cost_fun=mse,
    cost_der=mse_der,
    optimizer=Adam(gamma=0.01),
    seed=2026)

history100 = net100.train(
    X_train_s, 
    y_train, 
    epochs=500,
    batch_size=14,
    rng=np.random.default_rng(2026))

yn100_plot = net100.predict((x_plot - x_mean)/x_std) #scaling 

fig, ax = line_plot(
    x_plot,
    {
        "True Runge function": y_true,
        "OLS, degree 8": y_ols_plot,
        "NN, 1×100 sigmoid": yn100_plot,
    },
    xlabel="$x$",
    ylabel="$y$",
    data=(X_train, y_train),
    name="b100_nn_vs_ols")

y_pred_n100 = net100.predict(X_test_s)
mse_n100 = mse(y_pred_n100, y_test)
print(f"NN100 test MSE: {mse_n100:.6f}")

epochs = np.arange(1, len(history100) + 1)
fig, ax = line_plot(epochs, {"Adam": history100},
                    xlabel="Epoch", ylabel="Training MSE",
                    name="b100_learning_curve")

Notes on plots and mse:
- mse is higher ? 
- nn now has a worse fit ? 
- leanring curve is way more spikey (too big learning rate?) 


In [ ]:
structure = {
    "2×50":  [50, 50, 1],
    "2×100": [100, 100, 1]}

results = {}
for name, sizes in structure.items():
    nodes = len(sizes) - 1
    net = NeuralNetwork(
        network_input_size=1,
        layer_output_sizes=sizes,
        activation_funcs=[sigmoid] * nodes + [identity],
        activation_ders=[sigmoid_der] * nodes + [identity_der],
        cost_fun=mse,
        cost_der=mse_der,
        optimizer=Adam(gamma=0.01),     # new optimizer for every network
        seed=2026)
    history = net.train(X_train_s, y_train, epochs=500, batch_size=14,
                        rng=np.random.default_rng(2026))
    results[name] = {
        "test_mse": mse(net.predict(X_test_s), y_test),
        "history": history,
        "plot": net.predict((x_plot - x_mean) / x_std)}


fig, ax = line_plot(
    x_plot,
    {
        "True Runge function": y_true,
        "OLS, degree 8": y_ols_plot,
        "NN 2×50":  results["2×50"]["plot"],
    },
    xlabel="$x$",
    ylabel="$y$",
    data=(X_train, y_train),
    name="b_2x50_fit")

fig, ax = line_plot(
    x_plot,
    {
        "True Runge function": y_true,
        "OLS, degree 8": y_ols_plot,
        "NN 2×100": results["2×100"]["plot"],
    },
    xlabel="$x$",
    ylabel="$y$",
    data=(X_train, y_train),
    name="b_2x100_fit")

# Test MSE for the two-layer networks
for name, res in results.items():
    print(f"NN {name} test MSE: {res['test_mse']:.6f}")

# Learning curves for both networks in one plot
epochs = np.arange(1, len(results["2×50"]["history"]) + 1)
fig, ax = line_plot(
    epochs,
    {
        "NN 2×50":  results["2×50"]["history"],
        "NN 2×100": results["2×100"]["history"],
    },
    xlabel="Epoch",
    ylabel="Training MSE",
    name="b_2layers_learning_curve")

In [ ]:
sigma = 0.1
noise_floor = sigma**2

table = {
    "OLS, degree 8": mse_ols,
    "NN 1×50":       mse_n50,
    "NN 1×100":      mse_n100,
    "NN 2×50":       results["2×50"]["test_mse"],
    "NN 2×100":      results["2×100"]["test_mse"]}
table = {model: float(m) for model, m in table.items()}

df = pd.DataFrame({
    "Test MSE": table.values(),
    "Above noise floor": [m - noise_floor for m in table.values()],
}, index=table.keys())

display(df.round(4))
print(df.to_latex(float_format="%.4f"))

### Learning rate analysis 

In [ ]:
# Split the original training data into training and validation sets
X_train_nn, X_val, y_train_nn, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=2026)

# Scale using the new neural network training set
X_train_nn_s, X_val_s, x_mean_nn, x_std_nn = scale_data(
    X_train_nn, X_val)

# Scale the original test set using the same parameters
X_test_nn_s = (X_test - x_mean_nn) / x_std_nn

gammas = np.logspace(-4, 0, 9)
n_train = len(X_train_nn_s)

#Dictionary with the optimizers 
optimizers = {
    "GD": (GD, n_train),
    "SGD": (GD, 14),
    "RMSprop": (RMSprop, 14),
    "Adam": (Adam, 14)}

#loop so that lr_results saves both history and mse
lr_results = {}
for opt_name, (opt_class, batch_size) in optimizers.items():
    for gamma in gammas:
        net = NeuralNetwork(
            network_input_size=1,
            layer_output_sizes=[50, 50, 1],
            activation_funcs=[sigmoid, sigmoid, identity],
            activation_ders=[sigmoid_der, sigmoid_der, identity_der],
            cost_fun=mse,
            cost_der=mse_der,
            optimizer=opt_class(gamma=gamma),
            seed=2026)

        history = net.train(
            X_train_nn_s,
            y_train_nn,
            epochs=500,
            batch_size=batch_size,
            rng=np.random.default_rng(2026))

        lr_results[(opt_name, gamma)] = {
            "val_mse": float(mse(net.predict(X_val_s), y_val)),
            "test_mse": float(mse(net.predict(X_test_nn_s), y_test)), 
            "history": history}

In [ ]:
noise_floor = 0.1**2
diverge_limit = 1.0

curves = {}
diverged_gammas = {}

for opt_name in optimizers:
    mses = np.array([lr_results[(opt_name, g)]["val_mse"] for g in gammas])
    diverged = ~np.isfinite(mses) | (mses > diverge_limit)

    curves[opt_name] = np.where(diverged, np.nan, mses)
    diverged_gammas[opt_name] = np.array(gammas)[diverged]

    if np.any(~diverged):
        best = np.nanargmin(curves[opt_name])
        print(f"{opt_name}: best gamma = {gammas[best]:g}, "
              f"Validation MSE = {mses[best]:.4f}")
    else:
        print(f"{opt_name}: all learning rates diverged")

fig, ax = line_plot(
    np.array(gammas),
    curves,
    xlabel=r"Learning rate $\gamma$",
    ylabel="Validation MSE",
    logx=True,
    logy=True)

ax.axhline(noise_floor, color="gray", ls=":", label=r"Noise floor $\sigma^2$")

# Mark diverged runs with crosses
colors = {line.get_label(): line.get_color() for line in ax.get_lines()}
top = ax.get_ylim()[1]

for opt_name, gs in diverged_gammas.items():
    for g in gs:
        ax.plot(g, top, "x", color=colors[opt_name], ms=9, clip_on=False)

ax.legend(fontsize=8)
save_fig("b_lr_val_mse")


In [ ]:
best_curves = {}

for opt_name in optimizers:
    mses = np.array([lr_results[(opt_name, gamma)]["val_mse"] for gamma in gammas])
    diverged = ~np.isfinite(mses) | (mses > diverge_limit)
    valid_mses = np.where(diverged, np.nan, mses)

    best = np.nanargmin(valid_mses)
    best_gamma = gammas[best]

    history = lr_results[(opt_name, best_gamma)]["history"]
    best_curves[rf"{opt_name}, $\gamma$ = {best_gamma:2g}"] = history

epochs = np.arange(1, len(next(iter(best_curves.values()))) + 1)

fig, ax = line_plot(
    epochs,
    best_curves,
    xlabel="Epoch",
    ylabel="Training MSE",
    logy=True,
    hlines={r"Noise floor $\sigma^2$": noise_floor},
    name="b_lr_learning_curves")


In [ ]:
table_results = {}

for opt_name in optimizers:
    mses = np.array([lr_results[(opt_name, g)]["val_mse"] for g in gammas])
    diverged = ~np.isfinite(mses) | (mses > diverge_limit)

    valid_mses = np.where(diverged, np.nan, mses)
    best = np.nanargmin(valid_mses)

    table_results[opt_name] = {
    "Best learning rate": gammas[best],
    "Validation MSE": mses[best],
    "Test MSE": lr_results[(opt_name, gammas[best])]["test_mse"]}

df = pd.DataFrame.from_dict(table_results, orient="index")
df.index.name = "Optimizer"

display(df.round(4))
print(df.to_latex(float_format="%.4f"))

In [ ]:
# Compare GD and SGD with the same number of weight updates:
# SGD with batch size 14 takes 6 updates per epoch, full-batch GD takes 1.
steps_per_epoch = len(make_batches(n_train, 14, np.random.default_rng(2026)))
n_updates = 500 * steps_per_epoch   # 3000

gd_comparison = {}
for gamma in gammas:
    net_gd = NeuralNetwork(
        network_input_size=1,
        layer_output_sizes=[50, 50, 1],
        activation_funcs=[sigmoid, sigmoid, identity],
        activation_ders=[sigmoid_der, sigmoid_der, identity_der],
        cost_fun=mse,
        cost_der=mse_der,
        optimizer=GD(gamma=gamma),
        seed=2026)

    history = net_gd.train(X_train_nn_s, y_train_nn, epochs=n_updates,
                           batch_size=n_train, rng=np.random.default_rng(2026))

    gd_comparison[gamma] = {
        "val_mse": float(mse(net_gd.predict(X_val_s), y_val)),
        "history": history[steps_per_epoch - 1::steps_per_epoch]} 

print(f"Validation MSE, {n_updates} updates for GD and SGD:")
for gamma in gammas:
    print(f"gamma = {gamma:<8.3g}: "
          f"GD ({n_updates} updates) = {gd_comparison[gamma]['val_mse']:.4f}, "
          f"SGD ({n_updates} updates) = {lr_results[('SGD', gamma)]['val_mse']:.4f}, "
          f"GD (500 updates) = {lr_results[('GD', gamma)]['val_mse']:.4f}")

g_cmp = gammas[np.argmin(np.abs(gammas - 0.1))]
updates_axis = np.arange(1, 501) * steps_per_epoch      # both histories: one value per SGD epoch

fig, ax = line_plot(
    updates_axis,
    {rf"GD (full batch), $\gamma$ = {g_cmp:g}": gd_comparison[g_cmp]["history"],
     rf"SGD (batch size 14), $\gamma$ = {g_cmp:g}": lr_results[("SGD", g_cmp)]["history"]},
    xlabel="Weight updates",
    ylabel="Training MSE",
    logy=True,
    hlines={r"Noise floor $\sigma^2$": noise_floor},
    name="b_gd_vs_sgd_equal_updates")

In [ ]:
max_epochs = 2000

epoch_results = {}

for opt_name, (opt_class, batch_size) in optimizers.items():

    # Find the best learning rate from the previous experiment
    mses = np.array([
        lr_results[(opt_name, gamma)]["val_mse"]
        for gamma in gammas])

    diverged = ~np.isfinite(mses) | (mses > diverge_limit)
    valid_mses = np.where(diverged, np.nan, mses)

    if np.all(diverged):
        print(f"{opt_name}: all learning rates diverged")
        continue

    best_gamma = gammas[np.nanargmin(valid_mses)]

    # Create a fresh network
    net_epochs = NeuralNetwork(
        network_input_size=1,
        layer_output_sizes=[50, 50, 1],
        activation_funcs=[sigmoid, sigmoid, identity],
        activation_ders=[sigmoid_der, sigmoid_der, identity_der],
        cost_fun=mse,
        cost_der=mse_der,
        optimizer=opt_class(gamma=best_gamma),
        seed=2026)

    rng = np.random.default_rng(2026)
    train_mse = []
    val_mse = []

    # Train one epoch at a time and evaluate on the validation set
    for epoch in range(max_epochs):
        history = net_epochs.train(
            X_train_nn_s,
            y_train_nn,
            epochs=1,
            batch_size=batch_size,
            rng=rng)

        train_mse.append(history[-1])
        val_mse.append(
            float(mse(net_epochs.predict(X_val_s), y_val)))

    # store the results
    best_epoch = int(np.argmin(val_mse)) + 1
    epoch_results[opt_name] = {
        "gamma": best_gamma,
        "train_mse": train_mse,
        "val_mse": val_mse,
        "best_epoch": best_epoch}
    # print the epoch with the lowest validation MSE
    print(f"{opt_name} (gamma = {best_gamma:.2g}): lowest validation MSE "
          f"{val_mse[best_epoch - 1]:.4f} at epoch {best_epoch}, "
          f"{val_mse[499]:.4f} at epoch 500, {val_mse[-1]:.4f} at epoch {max_epochs}")

epoch_curves = {
    rf"{opt_name}, $\gamma$ = {results['gamma']:.2g}": results["val_mse"]
    for opt_name, results in epoch_results.items()}

epochs = np.arange(1, max_epochs + 1)

fig, ax = line_plot(
    epochs,
    epoch_curves,
    xlabel="Epoch",
    ylabel="Validation MSE",
    logy=True,
    # FIXED: noise floor as reference, as in the other plots
    hlines={r"Noise floor $\sigma^2$": noise_floor},
    name="b_epoch_validation_mse")

In [ ]:
best_gamma_final = best_gamma = table_results["Adam"]["Best learning rate"] 

net_final = NeuralNetwork(
    network_input_size=1,
    layer_output_sizes=[50, 50, 1],
    activation_funcs=[sigmoid, sigmoid, identity],
    activation_ders=[sigmoid_der, sigmoid_der, identity_der],
    cost_fun=mse,
    cost_der=mse_der,
    optimizer=Adam(gamma=best_gamma_final),
    seed=2026)

net_final.train(X_train_s, y_train, epochs=500, batch_size=14,
                rng=np.random.default_rng(2026))

mse_final = float(mse(net_final.predict(X_test_s), y_test))
print(f"OLS, degree 8:              test MSE = {mse_ols:.4f}")
print(f"NN 2×50, Adam, γ = {best_gamma_final:g}: test MSE = {mse_final:.4f}")

fig, ax = line_plot(
    x_plot,
    {"True Runge function": y_true,
     f"OLS, degree 8 (test MSE {mse_ols:.4f})": y_ols_plot,
     rf"NN 2×50, Adam, $\gamma$ = {best_gamma_final:g} (test MSE {mse_final:.4f})":
         net_final.predict((x_plot - x_mean) / x_std)},
    xlabel="$x$", ylabel="$y$",
    data=(X_train, y_train),
    name="b_final_ols_vs_nn")
ax.legend(loc="lower left")
save_fig("b_final_ols_vs_nn")

In [ ]:
net_final2 = NeuralNetwork(
    network_input_size=1,
    layer_output_sizes=[50, 50, 1],
    activation_funcs=[sigmoid, sigmoid, identity],
    activation_ders=[sigmoid_der, sigmoid_der, identity_der],
    cost_fun=mse,
    cost_der=mse_der,
    optimizer=Adam(gamma=best_gamma_final),
    seed=2026)

net_final2.train(X_train_s, y_train, epochs=1000, batch_size=14,
                rng=np.random.default_rng(2026))

mse_final2 = float(mse(net_final2.predict(X_test_s), y_test))
print(f"OLS, degree 8:              test MSE = {mse_ols:.4f}")
print(f"NN 2×50, Adam, γ = {best_gamma_final:g}: test MSE = {mse_final2:.4f}")

fig, ax = line_plot(
    x_plot,
    {"True Runge function": y_true,
     f"OLS, degree 8 (test MSE {mse_ols:.4f})": y_ols_plot,
     rf"NN 2×50, Adam, $\gamma$ = {best_gamma_final:g} (test MSE {mse_final2:.4f})":
         net_final2.predict((x_plot - x_mean) / x_std)},
    xlabel="$x$", ylabel="$y$",
    data=(X_train, y_train),
    name="b_final2_ols_vs_nn")
ax.legend(loc="lower left")
save_fig("b_final2_ols_vs_nn")